##### ARTI 560 - Computer Vision

## Image Classification with Vision Transformer (ViT) - Exercise

### Objective

In this exercise, you will test the pretrained Vision Transformer (ViT) model on 5 real-world images that you find online.

You will:

1. Download 5 images for different classes in [ImageNet](https://github.com/Waikato/wekaDeeplearning4j/blob/master/docs/user-guide/class-maps/IMAGENET.md).

2. Load the ImageNet class names from a [text file](https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt).

3. Use ViT to predict the class for each image.

4. Record whether the prediction was correct.

#### Important Note

For this exercise, you MUST use the following KerasHub components:

- [keras_hub.models.ViTImageClassifier](https://keras.io/keras_hub/api/models/vit/vit_image_classifier/)

- [keras_hub.models.ViTImageClassifierPreprocessor](https://keras.io/keras_hub/api/models/vit/vit_image_classifier_preprocessor/)

This ensures your input preprocessing (resizing + normalization) matches what the pretrained ViT model expects.

Do not replace the preprocessor with manual normalization (such as dividing by 255), because it may produce incorrect predictions.

In [ ]:
import os
import numpy as np
import requests
import keras_hub
from PIL import Image

# 1. Load ImageNet class names
classes_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(classes_url, timeout=30)
imagenet_classes = response.text.splitlines()

# 2. Load model + preprocessor
model_name = "vit_base_patch16_224_imagenet"
classifier = keras_hub.models.ViTImageClassifier.from_preset(model_name)
preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(model_name)

# 3. Image paths in Colab (after upload)
image_paths = [
    "download.jpg",
    "download (1).jpg",
    "download (2).jpg",
    "download (3).jpg",
    "download (4).jpg"
]

# 4. Load all images
images = []
valid_paths = []

for path in image_paths:
    if os.path.exists(path):
        img = Image.open(path).convert("RGB")
        images.append(np.array(img))
        valid_paths.append(path)
    else:
        print(f"File not found: {path}")

# 5. Convert to batch
images = np.array(images, dtype=object)

# 6. Preprocess + predict
processed = preprocessor(images)
predictions = classifier.predict(processed, verbose=0)

# 7. Print results
print(f"{'Image File':<35} | {'Prediction'}")
print("-" * 60)

for i, path in enumerate(valid_paths):
    predicted_idx = np.argmax(predictions[i])
    predicted_label = imagenet_classes[predicted_idx]
    print(f"{os.path.basename(path):<35} | {predicted_label}")

In [1]:
import tensorflow as tf
import keras_hub
import numpy as np
import pandas as pd
import requests
from PIL import Image
from io import BytesIO

# 1) ImageNet class names
classes_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_classes = requests.get(classes_url, timeout=30).text.strip().split("\n")

# 2) Load ViT preprocessor + model
preprocessor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
    "vit_base_patch16_224_imagenet"
)
model = keras_hub.models.ViTImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet"
)

# 3) 5 images (URLs) 
samples = [
    {"true_label": "pizza",      "url": "https://upload.wikimedia.org/wikipedia/commons/d/d3/Supreme_pizza.jpg"},
    {"true_label": "lemon",      "url": "https://upload.wikimedia.org/wikipedia/commons/c/c8/Lemon.jpg"},
    {"true_label": "orange",     "url": "https://upload.wikimedia.org/wikipedia/commons/c/c4/Orange-Fruit-Pieces.jpg"},
    {"true_label": "strawberry", "url": "https://upload.wikimedia.org/wikipedia/commons/2/29/PerfectStrawberry.jpg"},
    {"true_label": "broccoli",   "url": "https://upload.wikimedia.org/wikipedia/commons/0/03/Broccoli_and_cross_section_edit.jpg"},
]


# Session + headers 
session = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
    "Referer": "https://www.google.com/",
}

def load_image_from_url(url: str) -> np.ndarray:
    resp = session.get(url, headers=headers, timeout=30)
    resp.raise_for_status()
    img = Image.open(BytesIO(resp.content)).convert("RGB")
    return np.array(img, dtype=np.uint8)

results = []

for i, s in enumerate(samples, start=1):
    try:
        img = load_image_from_url(s["url"])

        x = preprocessor(img)
        x = tf.expand_dims(x, axis=0)

        logits = model(x)
        probs = tf.nn.softmax(logits, axis=-1).numpy()[0]

        top1 = int(np.argmax(probs))
        pred_label = imagenet_classes[top1]
        conf = float(probs[top1])

        correct = s["true_label"].lower() in pred_label.lower()

        results.append({
            "Image#": i,
            "True Label": s["true_label"],
            "Predicted Label (Top-1)": pred_label,
            "Confidence": round(conf, 4),
            "Correct?": "Yes" if correct else "No",
            "URL": s["url"],
        })
        
        
    except Exception as e:
        results.append({
            "Image#": i,
            "True Label": s["true_label"],
            "Predicted Label (Top-1)": "FAILED_TO_LOAD",
            "Confidence": None,
            "Correct?": "No",
            "URL": s["url"],
        })
        print(f"Image {i} failed to load: {e}")

df = pd.DataFrame(results)
df


d:\farahAnaconda3\envs\cv_lab\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.74k/1.74k [00:00<00:00, 555kB/s]


TypeError: <class 'keras_hub.src.models.vit.vit_image_classifier_preprocessor.ViTImageClassifierPreprocessor'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras_hub.src.models.vit.vit_image_classifier_preprocessor', 'class_name': 'ViTImageClassifierPreprocessor', 'config': {'name': 'vi_t_image_classifier_preprocessor', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'image_converter': {'module': 'keras_hub.src.models.vit.vit_image_converter', 'class_name': 'ViTImageConverter', 'config': {'name': 'vi_t_image_converter', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'image_size': [224, 224], 'scale': [0.00784313725490196, 0.0078

### Record Your Results

Fill the table below based on your results:

| Image File   | Predicted Label | True Label (What you searched) | Correct? (Yes/No) |
| ------------ | --------------- | ------------------------------ | ----------------- |
|              | ______          |                                | ______            |
|              | ______          |                                | ______            |
|              | ______          |                                | ______            |
|              | ______          |                                | ______            |
|              | ______          |                                | ______            |
